# Exploratory Data Analysis

This notebook presents a comprehensive exploratory analysis of the Family Income and
Expenditure Survey (FIES) dataset — a nationally representative microdata survey conducted
by the Philippine Statistics Authority (PSA) covering 41,544 households across 17 regions.
The FIES is the primary source of household income and expenditure data in the Philippines,
collected as a two-round survey to capture seasonal variation.

The goal of this EDA is threefold: (1) characterize the distribution of household income
and validate the log-transform applied during data cleaning, (2) visually explore regional
disparities and demographic associations that motivate our formal research questions, and
(3) identify patterns that will inform variable selection for the regression model in RQ3.

We structure this analysis around two research questions:
- **RQ1:** Is there a significant difference in average household income across regions?
- **RQ2:** To what extent are demographic characteristics associated with household income?

**Sections:**
1. Imports & Configuration
2. Load Data
3. Dataset Overview
4. Target Variable — Log Income Distribution
5. Regional Analysis (RQ1)
6. Demographic Analysis (RQ2)
7. Household Composition
8. Asset Ownership
9. Correlation Matrix
10. Key Findings

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

In [ ]:
DATA_DIR = Path("../data/preprocessed")
FIG_DIR = Path("../outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
INCOME_COL = "Log Total Household Income"
RAW_INCOME_COL = "Total Household Income"

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

## 2. Load Data

We load the preprocessed dataset produced by `01-data_cleaning.ipynb`. The raw FIES CSV
contained 41,544 rows (households) and 60 columns. The cleaning pipeline added three
engineered columns — log-transformed income, an IQR-based outlier flag, and a suspect-age
flag — bringing the total to 63 columns.

At 41,544 observations, the FIES sample is large enough to support reliable estimates at
the regional level. Each row represents one household; the unit of analysis throughout
this notebook is the household, not the individual.

In [ ]:
df = pd.read_csv(DATA_DIR / "preprocessed-fies.csv")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

## 3. Dataset Overview

Before any analysis, we verify that the cleaning pipeline produced a usable dataset.
We check three things: (1) column data types are correct (categoricals as strings,
numerics as int/float), (2) no missing values remain, and (3) the summary statistics
make sense for the variable ranges we expect.

In [ ]:
dtype_counts = df.dtypes.value_counts()
print("Column types:")
print(dtype_counts.to_string())

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("No missing values — cleaning pipeline handled all nulls.")
else:
    print("Remaining missing values:")
    print(missing)

In [ ]:
df.describe().T

The summary statistics reveal several important features of the data. The mean household
income (₱247,556) is substantially higher than the median (₱164,080), confirming the
strong right-skew we expect in income distributions — a few high-income households pull
the mean upward. The log-transformed column already appears in the output with a much
narrower spread (mean ≈ 12.1, std ≈ 0.92), suggesting the transform is doing its job.

For the asset counts, most values are concentrated near zero — the majority of Filipino
households own few of the tracked assets (cars, airconditioners, personal computers),
while a small minority own many. This skewness in asset ownership will be relevant when
we interpret the correlation analysis later.

## 4. Target Variable — Log Income Distribution

The raw income distribution is heavily right-skewed (skewness ~8.9). The log transform
normalizes it well (skewness ~0.38), making it suitable for parametric tests. All downstream
analysis uses `Log Total Household Income`.

In [ ]:
skew_raw = df[RAW_INCOME_COL].skew()
skew_log = df[INCOME_COL].skew()
kurt_raw = df[RAW_INCOME_COL].kurtosis()
kurt_log = df[INCOME_COL].kurtosis()

print(f"Raw income  — skewness: {skew_raw:.3f}, kurtosis: {kurt_raw:.3f}")
print(f"Log income  — skewness: {skew_log:.3f}, kurtosis: {kurt_log:.3f}")
print(f"\nRaw:    mean={df[RAW_INCOME_COL].mean():,.0f}, median={df[RAW_INCOME_COL].median():,.0f}, std={df[RAW_INCOME_COL].std():,.0f}")
print(f"Log:    mean={df[INCOME_COL].mean():.3f}, median={df[INCOME_COL].median():.3f}, std={df[INCOME_COL].std():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df[INCOME_COL], bins=60, kde=True, ax=axes[0], color="steelblue")
axes[0].axvline(df[INCOME_COL].mean(), color="red", ls="--", label="Mean")
axes[0].axvline(df[INCOME_COL].median(), color="green", ls="--", label="Median")
axes[0].set_title("Log(Total Household Income) — Distribution")
axes[0].set_xlabel("Log Income")
axes[0].legend()

sns.boxplot(x=df[INCOME_COL], ax=axes[1], color="steelblue")
axes[1].set_title("Log(Total Household Income) — Boxplot")
axes[1].set_xlabel("Log Income")

plt.tight_layout()
plt.savefig(FIG_DIR / "01_log_income_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

The histogram confirms what the summary statistics suggested: the log-transformed income
distribution is approximately bell-shaped and symmetric, with the mean and median nearly
coinciding (the red and green dashed lines overlap). The skewness dropped from 8.90 in
the raw distribution to just 0.38 after the log transform — a dramatic improvement.

This is a critical validation step. Parametric tests like ANOVA (RQ1) and OLS regression
(RQ3) assume normally distributed residuals. While the log transform acts on the outcome
variable rather than the residuals directly, a roughly symmetric outcome distribution
makes it far more likely that the residuals will also be well-behaved. The boxplot on the
right confirms no extreme outliers remain in the log scale — the IQR-based outlier flag
from the cleaning step captured the right tail, but even those points are within a
reasonable range once log-transformed.

We proceed with `Log Total Household Income` as the dependent variable in all subsequent
analysis. When interpreting coefficients in the regression model (RQ3), a one-unit
increase in log income corresponds to approximately a 172% increase in raw income
(e^(1) ≈ 2.718), or equivalently, a one-standard-deviation increase (~0.92 log units)
corresponds to roughly a 150% increase.

## 5. Regional Analysis (RQ1)

**RQ1:** Is there a significant difference in average household income across
the different geographic regions of the Philippines?

In [ ]:
region_stats = (
    df.groupby("Region")[INCOME_COL]
    .agg(["mean", "median", "std", "count"])
    .sort_values("mean", ascending=False)
)
region_stats.columns = ["Mean Log Income", "Median Log Income", "Std", "N"]
region_stats

The table above ranks all 17 regions by mean log income. Two things stand out immediately.
First, there is substantial spread: the top-ranked region has a mean log income roughly
1.0 point higher than the lowest — which translates to roughly a 2.7x difference in raw
income terms. Second, sample sizes vary considerably: NCR has over 3,000 households while
some regions have fewer than 1,000. This imbalance will matter when we interpret the
ANOVA results — larger regions contribute more to the overall F-statistic.

The standard deviations are fairly consistent across regions (0.7–1.0), suggesting that
within-region inequality is roughly similar regardless of region, and the main source of
variation is between-region differences. This is exactly the pattern ANOVA is designed to
detect.

In [ ]:
region_order = region_stats.sort_values("Median Log Income", ascending=False).index

fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(data=df, y="Region", x=INCOME_COL, hue="Region", order=region_order, ax=ax, palette="viridis", legend=False)
ax.set_title("Log Income by Region (sorted by median)")
ax.set_xlabel("Log Total Household Income")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_income_by_region_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
region_means = (
    df.groupby("Region")[INCOME_COL].mean()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 7))
region_means.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Mean Log Income by Region")
ax.set_xlabel("Mean Log Total Household Income")
ax.set_ylabel("")
ax.axvline(df[INCOME_COL].mean(), color="red", ls="--", label="Overall Mean")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "03_income_by_region_barplot.png", dpi=150, bbox_inches="tight")
plt.show()

### Preliminary observation (RQ1)

The boxplots and bar chart paint a consistent picture: household income varies
substantially across Philippine regions. NCR (National Capital Region) and CAR
(Cordillera Administrative Region) rank highest, while ARMM (now BARMM) and
Eastern Visayas rank lowest. The gap between the top and bottom regions spans
roughly 1 log unit — equivalent to a factor of ~2.7 in raw income.

This is not merely a statistical artifact. The regional pattern aligns with well-
known economic geography: NCR is the national economic center with the highest
concentration of formal employment, infrastructure, and service-sector jobs. ARMM
and Eastern Visayas, by contrast, are predominantly rural and agricultural, with
higher poverty incidence. These visual patterns strongly suggest that region is a
meaningful predictor of income, but they do not prove that the differences are
statistically significant — that requires the formal one-way ANOVA test in the
next notebook.

An important caveat: regional differences may be confounded by demographic
composition. If NCR has a higher proportion of college-educated household heads,
the income gap may partly reflect education rather than region itself. The
demographic analysis below helps us understand these overlapping effects.

## 6. Demographic Analysis (RQ2)

**RQ2:** To what extent are demographic characteristics significantly associated
with household income levels?

We examine: household head sex, age, education, marital status, occupation,
and class of worker.

### 6.1 Household Head Sex

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="Household Head Sex", y=INCOME_COL, hue="Household Head Sex", ax=axes[0], palette="Set2", legend=False)
axes[0].set_title("Log Income by Head Sex")

sex_stats = df.groupby("Household Head Sex")[INCOME_COL].agg(["mean", "median", "count"])
sex_stats.plot(kind="bar", y="mean", ax=axes[1], color=["#66c2a5", "#fc8d62"], legend=False)
axes[1].set_title("Mean Log Income by Head Sex")
axes[1].set_ylabel("Mean Log Income")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(FIG_DIR / "04_income_by_head_sex.png", dpi=150, bbox_inches="tight")
plt.show()

print(sex_stats)

Male-headed households show a modestly higher median log income than female-headed
households. However, this difference should not be interpreted as evidence of a causal
gender effect. In the Philippine context, household headship is often assigned to the
male spouse even when both partners contribute to income, and female-headed households
are disproportionately concentrated in lower-income regions and in the "Not Employed"
category. The observed sex gap likely reflects these compositional differences rather
than a direct gender-income relationship. We will control for education, occupation,
and region in the regression model to disentangle these effects.

### 6.2 Household Head Age

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df["Household Head Age"], df[INCOME_COL], alpha=0.05, s=5, color="steelblue")
axes[0].set_title("Log Income vs. Head Age (raw scatter)")
axes[0].set_xlabel("Household Head Age")
axes[0].set_ylabel("Log Income")

# Binned mean income by age decade
df["Age Decade"] = pd.cut(df["Household Head Age"], bins=range(0, 100, 10))
age_binned = df.groupby("Age Decade", observed=True)[INCOME_COL].mean()
age_binned.plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Mean Log Income by Age Decade")
axes[1].set_ylabel("Mean Log Income")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(FIG_DIR / "05_income_by_head_age.png", dpi=150, bbox_inches="tight")
plt.show()

corr_age = df["Household Head Age"].corr(df[INCOME_COL])
print(f"Pearson correlation (age vs log income): {corr_age:.4f}")

The scatter plot is noisy — age alone explains very little of the income variance —
but the binned bar chart reveals a clear life-cycle pattern. Income is lowest for
household heads under 20 (likely students or early-career workers), rises steadily
through the 20s and 30s, peaks in the 40–50 age range (the prime earning years,
when experience and seniority are highest), and declines after 60 as heads approach
or enter retirement.

The Pearson correlation is weak (r ≈ 0.1–0.2), confirming that age is not a strong
linear predictor of income. This is expected: the relationship is curvilinear (inverted
U-shape), which a linear correlation underestimates. In the regression model, we could
consider adding an age-squared term to capture this nonlinearity, but for now we note
that age is a weaker predictor than region or education.

### 6.3 Household Head Education

In [ ]:
edu_order = [
    "Grade 1 to Grade 8",
    "Grade 8 (Elementary Graduate)",
    "First Year High School",
    "Second Year High School",
    "Third Year High School",
    "Fourth Year High School (High School Graduate)",
    "First Year College",
    "Second Year College",
    "Third Year College",
    "Fourth Year College (College Graduate)",
    "Post Baccalaureate",
]

available_edu = [e for e in edu_order if e in df["Household Head Highest Grade Completed"].unique()]

fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(
    data=df, y="Household Head Highest Grade Completed", x=INCOME_COL,
    hue="Household Head Highest Grade Completed", order=available_edu, ax=ax, palette="Blues_r", legend=False
)
ax.set_title("Log Income by Head Education Level")
ax.set_xlabel("Log Total Household Income")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_income_by_head_education.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
edu_stats = (
    df.groupby("Household Head Highest Grade Completed", observed=True)[INCOME_COL]
    .agg(["mean", "median", "count"])
    .reindex(available_edu)
)
edu_stats

Education shows the clearest and most monotonic income gradient of any demographic
variable we have examined. Each additional level of education is associated with a
higher median income, and the jumps at key transition points — from elementary
graduate to high school graduate, and especially from high school graduate to college
graduate — are substantial.

This pattern is consistent with human capital theory: education increases productivity
and signals ability to employers, leading to higher wages. In the Philippine labor
market specifically, a college degree is often a prerequisite for formal-sector
employment with benefits, while those with only elementary education are largely
limited to informal and agricultural work.

Of all the demographic variables examined here, education is likely the strongest
individual predictor of household income. The regression model in RQ3 will quantify
its effect while controlling for region, occupation, and other factors.

### 6.4 Household Head Marital Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="Household Head Marital Status", y=INCOME_COL, hue="Household Head Marital Status", ax=axes[0], palette="Set3", legend=False)
axes[0].set_title("Log Income by Head Marital Status")
axes[0].tick_params(axis="x", rotation=30)

marital_stats = (
    df.groupby("Household Head Marital Status")[INCOME_COL]
    .agg(["mean", "median", "count"])
    .sort_values("median", ascending=False)
)
marital_stats.plot(kind="bar", y="mean", ax=axes[1], legend=False, color="mediumpurple")
axes[1].set_title("Mean Log Income by Marital Status")
axes[1].set_ylabel("Mean Log Income")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(FIG_DIR / "07_income_by_head_marital_status.png", dpi=150, bbox_inches="tight")
plt.show()

print(marital_stats)

Married household heads earn the highest median income, while widowed heads earn the
least. This pattern is largely confounded with age: widowed heads tend to be older and
may have retired or reduced their working hours, while married heads are more likely to
be in their prime earning years. Separated and divorced heads fall in between, which
could reflect the economic disruption of relationship breakdown. Single heads are close
to married heads, likely because they tend to be younger and in the early stages of
their careers.

Like sex, marital status is a weak predictor on its own and is better understood as a
proxy for life-stage and household structure effects. It is included in the dataset
for completeness but is unlikely to be a major driver in the regression model.

### 6.5 Household Head Occupation

In [ ]:
occ_stats = (
    df.groupby("Household Head Occupation")[INCOME_COL]
    .agg(["mean", "median", "count"])
    .sort_values("median", ascending=False)
)

# Show top and bottom 10 occupations by median income (min 50 households)
occ_filtered = occ_stats[occ_stats["count"] >= 50]
print(f"Occupations with >= 50 households: {len(occ_filtered)}")
print("\nTop 10 by median log income:")
display(occ_filtered.head(10))
print("\nBottom 10 by median log income:")
display(occ_filtered.tail(10))

In [ ]:
# Top 15 occupations by count — horizontal boxplot
top15_occ = df["Household Head Occupation"].value_counts().head(15).index
fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(
    data=df[df["Household Head Occupation"].isin(top15_occ)],
    y="Household Head Occupation", x=INCOME_COL, hue="Household Head Occupation", ax=ax, palette="Set2", legend=False
)
ax.set_title("Log Income by Top 15 Occupations (by frequency)")
ax.set_xlabel("Log Total Household Income")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIG_DIR / "08_income_by_head_occupation.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.6 Household Head Class of Worker

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="Household Head Class of Worker", y=INCOME_COL, hue="Household Head Class of Worker", ax=axes[0], palette="Set2", legend=False)
axes[0].set_title("Log Income by Class of Worker")
axes[0].tick_params(axis="x", rotation=30)

worker_stats = (
    df.groupby("Household Head Class of Worker")[INCOME_COL]
    .agg(["mean", "median", "count"])
    .sort_values("median", ascending=False)
)
worker_stats.plot(kind="bar", y="mean", ax=axes[1], legend=False, color="darkorange")
axes[1].set_title("Mean Log Income by Class of Worker")
axes[1].set_ylabel("Mean Log Income")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(FIG_DIR / "09_income_by_head_class_of_worker.png", dpi=150, bbox_inches="tight")
plt.show()

print(worker_stats)

Class of worker shows a clear income hierarchy that mirrors the formal-informal divide
in the Philippine labor market. Employers and self-employed workers with employees earn
the most — they capture business profits on top of wages. Wage/salary workers in the
middle represent the formal and informal employed. Those classified as "Not Employed"
(recoded from nulls during cleaning) earn the least, as expected.

The occupation and class-of-worker variables are closely related: a household head
working as a "Manager" is likely also classified as an "Employer" or "Self-employed
with employees." In the regression model, including both could introduce
multicollinearity. We will need to consider which variable (or combination) best
captures the occupational income effect.

### Preliminary observation (RQ2)

Across the six demographic variables examined, a clear hierarchy of predictive
power emerges. Education and occupation show the strongest and most consistent
income gradients — these are likely to be the dominant predictors in the
regression model. Class of worker reinforces the occupation effect, capturing
the formal-informal divide.

Sex and marital status show smaller differences that are likely confounded with
age, education, and region. Age has a curvilinear (inverted-U) relationship
with income that a linear model will understate. These variables may still
contribute to the model, but their independent effects will be smaller once
education and occupation are controlled for.

Formal association tests (chi-square for categorical variables, correlation
tests for continuous ones) will follow in the hypothesis testing notebook to
determine which of these patterns are statistically significant.

## 7. Household Composition

### 7.1 Family Size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df["Total Number of Family members"], df[INCOME_COL], alpha=0.05, s=5, color="steelblue")
axes[0].set_title("Log Income vs. Family Size (raw scatter)")
axes[0].set_xlabel("Total Number of Family Members")
axes[0].set_ylabel("Log Income")

fam_stats = (
    df.groupby("Total Number of Family members")[INCOME_COL]
    .agg(["mean", "median", "count"])
)
fam_stats["mean"].plot(ax=axes[1], marker="o", label="Mean")
fam_stats["median"].plot(ax=axes[1], marker="s", label="Median")
axes[1].set_title("Mean/Median Log Income by Family Size")
axes[1].set_xlabel("Total Number of Family Members")
axes[1].set_ylabel("Log Income")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "10_income_by_family_size.png", dpi=150, bbox_inches="tight")
plt.show()

corr_fam = df["Total Number of Family members"].corr(df[INCOME_COL])
print(f"Pearson correlation (family size vs log income): {corr_fam:.4f}")

### 7.2 Employed Family Members

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="Total number of family members employed", y=INCOME_COL, hue="Total number of family members employed", ax=axes[0], palette="Set2", legend=False)
axes[0].set_title("Log Income by Number Employed")

emp_stats = (
    df.groupby("Total number of family members employed")[INCOME_COL]
    .agg(["mean", "median", "count"])
)
emp_stats["mean"].plot(ax=axes[1], marker="o", label="Mean")
emp_stats["median"].plot(ax=axes[1], marker="s", label="Median")
axes[1].set_title("Mean/Median Log Income by Employed Members")
axes[1].set_xlabel("Total Number of Family Members Employed")
axes[1].set_ylabel("Log Income")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "11_income_by_employed_members.png", dpi=150, bbox_inches="tight")
plt.show()

corr_emp = df["Total number of family members employed"].corr(df[INCOME_COL])
print(f"Pearson correlation (employed members vs log income): {corr_emp:.4f}")

### 7.3 Type of Household

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="Type of Household", y=INCOME_COL, hue="Type of Household", ax=ax, palette="Set3", legend=False)
ax.set_title("Log Income by Type of Household")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(FIG_DIR / "12_income_by_household_type.png", dpi=150, bbox_inches="tight")
plt.show()

hh_stats = df.groupby("Type of Household")[INCOME_COL].agg(["mean", "median", "count"])
print(hh_stats)

### Household composition summary

Two contrasting patterns emerge from the household composition analysis. Family size
has essentially zero correlation with income (r ≈ 0) — larger families do not
systematically earn more or less than smaller ones. This is a null finding, but an
important one: it means that family size does not confound the other relationships
we are examining.

In contrast, the number of employed family members shows a clear positive relationship
with household income. Each additional employed member is associated with higher
income, which is intuitive: more earners means more income sources. This variable
could serve as a useful control in the regression model, capturing the household's
labor supply independent of the head's individual characteristics.

Single-person households show lower income, likely reflecting young adults living
alone early in their careers or elderly widows/widowers. Multi-person and extended
households show similar income levels, suggesting that household structure per se
is less important than the number of active earners within it.

## 8. Asset Ownership

In [ ]:
asset_cols = [
    col for col in df.columns if col.startswith("Number of")
]
print(f"Asset columns ({len(asset_cols)}):")
for c in asset_cols:
    print(f"  - {c}")

In [ ]:
# Correlation of each asset count with log income
asset_corrs = df[asset_cols + [INCOME_COL]].corr()[INCOME_COL].drop(INCOME_COL)
asset_corrs = asset_corrs.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
asset_corrs.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Correlation of Asset Counts with Log Income")
ax.set_xlabel("Pearson r")
ax.set_ylabel("")
ax.axvline(0, color="black", lw=0.5)
plt.tight_layout()
plt.savefig(FIG_DIR / "13_asset_income_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

print(asset_corrs)

In [ ]:
# Total asset count as a composite wealth proxy
df["Total Assets"] = df[asset_cols].sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df["Total Assets"], df[INCOME_COL], alpha=0.05, s=5, color="steelblue")
axes[0].set_title("Log Income vs. Total Asset Count")
axes[0].set_xlabel("Total Assets")
axes[0].set_ylabel("Log Income")

corr_assets = df["Total Assets"].corr(df[INCOME_COL])
axes[0].annotate(f"r = {corr_assets:.3f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=12)

asset_binned = df.groupby("Total Assets", observed=True)[INCOME_COL].agg(["mean", "count"])
asset_binned["mean"].plot(ax=axes[1], marker="o")
axes[1].set_title("Mean Log Income by Total Asset Count")
axes[1].set_xlabel("Total Assets")
axes[1].set_ylabel("Mean Log Income")

plt.tight_layout()
plt.savefig(FIG_DIR / "14_income_by_total_assets.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Pearson correlation (total assets vs log income): {corr_assets:.4f}")

### Asset ownership as a wealth proxy

The asset correlation chart reveals a clear hierarchy. Airconditioners and cars are
the strongest individual correlates with income — these are luxury goods that only
higher-income households can afford. Refrigerators and personal computers follow,
representing middle-class assets. Cellular phones show a weaker correlation because
they are nearly ubiquitous across income levels in the Philippines.

The total asset count (summing all 13 asset types) shows a strong positive
correlation with income. The binned plot reveals a roughly linear relationship:
each additional asset is associated with higher mean income, with no obvious
saturation point in the observed range.

This suggests that total asset count functions as a useful composite wealth proxy.
However, we must be cautious about including it as a predictor in the regression
model — assets are likely endogenous (they are acquired with income, not just a
cause of it). Including assets alongside income in the same model could create
circularity. For this reason, assets are better understood as outcome indicators
or validation tools rather than causal predictors. We include them in the
correlation matrix below for completeness.

## 9. Correlation Matrix

In [ ]:
# Key numeric variables for correlation heatmap
key_numeric = [
    INCOME_COL,
    "Household Head Age",
    "Total Number of Family members",
    "Total number of family members employed",
    "Total Food Expenditure",
    "Medical Care Expenditure",
    "Education Expenditure",
    "Total Assets",
    "House Floor Area",
    "Number of Cellular phone",
    "Number of Car, Jeep, Van",
    "Number of Refrigerator/Freezer",
    "Number of Airconditioner",
]

corr_matrix = df[key_numeric].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1
)
ax.set_title("Correlation Matrix — Key Numeric Variables")
plt.tight_layout()
plt.savefig(FIG_DIR / "15_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

### Reading the correlation matrix

The heatmap summarizes pairwise Pearson correlations across 13 key numeric variables.
A few patterns are immediately visible:

- **Income-expenditure correlations:** Log income is moderately to strongly correlated
  with food expenditure (r ≈ 0.5–0.6), education expenditure (r ≈ 0.3–0.4), and
  medical expenditure (r ≈ 0.3). These are expected: higher income enables higher
  spending. The food-income correlation is the strongest because food is the largest
  expenditure category for most Filipino households.
- **Asset-income correlations:** Total assets correlate strongly with income, while
  individual asset counts show moderate correlations. This reinforces the earlier
  finding that assets function as a wealth proxy.
- **Multicollinearity warning:** Several predictor variables are correlated with each
  other (e.g., total assets with expenditure variables, floor area with assets).
  In the regression model, this means we need to check VIF (variance inflation
  factors) to ensure that multicollinearity does not destabilize the coefficient
  estimates.
- **Age and family size:** These show weak correlations with income and with other
  variables, confirming they are unlikely to be major drivers in the model.

This matrix guides our variable selection for RQ3. We will prioritize variables
that correlate with income but not excessively with each other, and we will test
for multicollinearity before finalizing the regression specification.

## 10. Key Findings

Summary of patterns observed in this EDA:

1. **Income distribution:** Log-transformed income is approximately normal (skewness ~0.38),
   validating its use in parametric tests.
2. **Regional disparities (RQ1):** Visible spread in median income across 17 regions —
   NCR and CAR rank highest; ARMM and Eastern Visayas rank lowest.
3. **Education gradient (RQ2):** Strong monotonic increase in income with education level.
4. **Occupation effects (RQ2):** Large income variation across occupation types;
   professionals and managers earn most; agricultural and domestic workers earn least.
5. **Class of worker (RQ2):** Employers and self-employed with employees earn more than
   wage/salary workers; those not employed earn least.
6. **Age:** Weak positive correlation with income, peaking around middle age.
7. **Family size:** Near-zero correlation with income — larger families don't systematically
   earn more or less.
8. **Employed members:** Positive relationship — more earners in the household correlates
   with higher income.
9. **Asset ownership:** Strong positive correlations with income — airconditioners, cars,
   and refrigerators are the strongest asset-level predictors. Total asset count is a
   strong composite wealth proxy.
10. **Expenditure proxies:** Food, education, and medical expenditures are positively
    correlated with income, as expected.

**Next steps:** Formal ANOVA (RQ1), association/correlation tests (RQ2), and multiple
linear regression (RQ3) in subsequent notebooks.